# 법인카드 이상거래 탐지 — Tier 0 vs Tier 0+1 비교 실험

**목적**: 1차 베이스라인(`모델링_v1/v2`)에서 Tier 0(14개)만으로 Isolation Forest를 돌렸을 때
PR-AUC 0.4711, recall@top5% 0.6300이 나왔다. 하이퍼파라미터 튜닝은 개선폭이 노이즈 수준(<1%)이라
포기했으므로(fold 표준편차 0.051 > 튜닝 개선폭 0.0012), 이번엔 **Tier 1(카드/거래 정산 메타데이터,
11개)을 추가했을 때 성능이 실제로 개선되는지**를 확인한다.

이 근거는 raw_payload 스키마 확정 논의(Open Issue #6-7)에도 쓰인다 — 개선폭이 크면 raw_payload에
카드 메타데이터를 반드시 포함시켜야 한다는 근거가 되고, 미미하면 Tier 0만으로도 충분하다는 근거가 된다.

**설계 원칙**
- 같은 노트북 안에서 Tier 0만 / Tier 0+1을 **동일한 fold, 동일한 하이퍼파라미터**로 평가해 공정하게 비교한다
  (베이스라인 노트북과 별도 실행이면 fold 셔플이나 환경 차이로 완벽히 같은 조건이라 보장하기 어려움)
- 하이퍼파라미터는 튜닝에서 개선 효과가 없었던 걸 확인했으므로 **베이스라인 값(n_estimators=200, max_samples='auto')을
  그대로 사용** — 지금 궁금한 건 "피처를 늘렸을 때 효과"이지 "하이퍼파라미터 효과"가 아니므로, 두 변수를 동시에
  바꾸면 어느 쪽 덕분에 성능이 변했는지 구분할 수 없다
- `test_df`는 이번에도 열지 않는다 (5단계 범위, fold 평가만)


## 1. 라이브러리 & 데이터 로드

In [1]:
import json
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score

pd.set_option('display.max_columns', 60)

DATA_DIR = '../.data/processed'
TRAIN_PATH = os.path.join(DATA_DIR, 'train_processed.csv')
TIERS_PATH = os.path.join(DATA_DIR, 'feature_tiers.json')

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
with open(TIERS_PATH, encoding='utf-8') as f:
    FEATURE_TIERS = json.load(f)

print(f"train_df: {train_df.shape}")
print(f"Tier 0: {len(FEATURE_TIERS['tier0_transaction_safe'])}개, "
      f"Tier 1: {len(FEATURE_TIERS['tier1_card_settlement_meta'])}개")


train_df: (1482969, 47)
Tier 0: 14개, Tier 1: 11개


## 2. dtype 재적용

Tier 0 재적용(요일/시간대구간 category, 거래일자 datetime, 월말여부 bool)에 더해, Tier 1에서 쓰는 컬럼 중
전처리 노트북 6절 `CATEGORICAL_COLS`에 있던 7개(`개인법인구분코드_회원`, `국내해외여부`, `승인거래코드`,
`승인발생경로코드`, `로고구분코드`, `일시불할부구분코드`, `카드구분코드`)를 category로, 4절에서 별도
`pd.Categorical`로 만들었던 `카드이용한도금액_구간`(순서형)을 다시 캐스팅한다.

In [2]:
# Tier 0
train_df['거래요일_한글'] = train_df['거래요일_한글'].astype('category')
train_df['시간대구간'] = train_df['시간대구간'].astype('category')
train_df['거래일자'] = pd.to_datetime(train_df['거래일자'])
train_df['월말여부'] = train_df['월말여부'].astype(bool)

# Tier 1 — 명목형(순서 없음)
TIER1_NOMINAL_CAT = [
    '개인법인구분코드_회원', '국내해외여부', '승인거래코드',
    '승인발생경로코드', '로고구분코드', '일시불할부구분코드', '카드구분코드',
]
for col in TIER1_NOMINAL_CAT:
    train_df[col] = train_df[col].astype('category')

# Tier 1 — 순서형 (전처리 4절: pd.Categorical(categories=[0, 1e6, 5e6, 1e7], ordered=True))
train_df['카드이용한도금액_구간'] = pd.Categorical(
    train_df['카드이용한도금액_구간'],
    categories=[0, 1_000_000, 5_000_000, 10_000_000],
    ordered=True,
)

print("Tier 1 범주형 컬럼별 고유값 개수 (원-핫 시 차원 폭발 여부 확인):")
for col in TIER1_NOMINAL_CAT:
    print(f"  {col}: {train_df[col].nunique(dropna=True)}개")


Tier 1 범주형 컬럼별 고유값 개수 (원-핫 시 차원 폭발 여부 확인):
  개인법인구분코드_회원: 2개
  국내해외여부: 2개
  승인거래코드: 5개
  승인발생경로코드: 4개
  로고구분코드: 7개
  일시불할부구분코드: 3개
  카드구분코드: 5개


## 3. 모델 입력 행렬 준비 함수 (Tier 0 / Tier 0+1 공용)

**[수정] 이전 실행에서 두 가지 문제가 확인되어 인코딩 방식을 바꾼다.**

1. **`dummy_na=True`를 무조건 켜뒀던 게 문제였다.** `거래요일_한글`·`시간대구간`처럼 애초에 NaN이 없는
   컬럼에도 "NaN 여부" 더미 컬럼이 만들어져, 정보가 전혀 없는 상수(전부 0) 컬럼이 섞여 들어갔다. Isolation
   Forest는 분기 피처를 **완전히 무작위로** 고르기 때문에, 이런 무의미한 컬럼이 하나라도 있으면 분기 기회가
   그쪽으로 낭비되어 실제 이상치 판별력이 떨어진다. → **실제로 NaN이 있는 컬럼에만** `dummy_na`를 켠다.
2. **Tier 1 범주형을 원-핫으로 풀었더니 컬럼이 21→62개로 3배 늘면서 오히려 성능이 60% 가까이 떨어졌다.**
   저카디널리티(2~7개)라도 원-핫으로 풀면 대부분 값이 2개뿐인 이진 컬럼이 되는데, 이진 컬럼은 어디서 분기하든
   결과가 거의 같아 판별력이 거의 없다. Isolation Forest가 무작위로 피처를 고르는 구조상, 이런 저정보 컬럼
   비중이 커질수록 진짜 유용한 연속형 피처(카드누적사용액, Zscore 등)가 뽑힐 확률이 희석된다.
   → **Tier 1 범주형은 원-핫 대신 순서 코드(`.cat.codes`)로 인코딩**해 컬럼 수를 늘리지 않는다. `거래요일_한글`·
   `시간대구간`(Tier 0)은 원래도 성능이 잘 나왔던 조합이라 그대로 원-핫을 유지한다.

**[참고] `할부가능개월수`**: 일시불 거래는 할부 개월 수 개념이 없어 NaN으로 기록된 것으로 보이므로, median이
아니라 **0(할부 없음)으로 채운다.**

In [3]:
NAN_FILL_COLS = ['사용자평균사용액_확장', '사용자표준편차_확장', '거래금액_Zscore_확장']
ZERO_FILL_COLS = ['할부가능개월수']  # 일시불 거래는 개념 자체가 없어 0(할부 없음)으로 채움 — median 부적절

# Tier 0에서 원래도 잘 작동했던 두 컬럼만 원-핫 유지. 나머지 범주형(Tier 1 포함)은 전부 순서 코드로 인코딩해
# 컬럼 수가 불필요하게 늘어나 Isolation Forest의 무작위 피처 선택이 희석되는 걸 방지한다.
TIER0_ONEHOT_COLS = ['거래요일_한글', '시간대구간']
TIER0_DROP_FROM_MODEL = ['거래일자', '거래연월']
ORDINAL_COLS = ['카드이용한도금액_구간']  # 순서형(구간 클수록 값도 큼) — codes가 순서 정보를 보존


def build_model_matrix(df, tier_cols, fill_values):
    '''주어진 피처 컬럼 목록(Tier 0 또는 Tier 0+1)을 Isolation Forest 입력용 수치 행렬로 변환.'''
    cols = [c for c in tier_cols if c not in TIER0_DROP_FROM_MODEL]
    X = df[cols].copy()

    # 1) 콜드스타트 확장 통계 NaN → median
    present_nan_cols = [c for c in NAN_FILL_COLS if c in X.columns]
    if present_nan_cols:
        X[present_nan_cols] = X[present_nan_cols].fillna(fill_values[present_nan_cols])

    # 1-1) 일시불이라 개념 자체가 없는 컬럼(할부가능개월수) → 0
    present_zero_cols = [c for c in ZERO_FILL_COLS if c in X.columns]
    if present_zero_cols:
        X[present_zero_cols] = X[present_zero_cols].fillna(0)

    # 2) bool 컬럼 → int
    for col in X.columns:
        if X[col].dtype == bool:
            X[col] = X[col].astype(int)

    # 3) 순서형 범주(카드이용한도금액_구간) → codes
    for col in ORDINAL_COLS:
        if col in X.columns:
            X[col] = X[col].cat.codes

    # 4) 나머지 category 컬럼: Tier0 지정 2개만 원-핫(실제 NaN 있을 때만 dummy_na), 그 외(Tier1 명목형)는 전부 codes
    remaining_cat_cols = [c for c in X.columns if str(X[c].dtype) == 'category']
    onehot_cols = [c for c in remaining_cat_cols if c in TIER0_ONEHOT_COLS]
    code_cols = [c for c in remaining_cat_cols if c not in TIER0_ONEHOT_COLS]

    for col in code_cols:
        X[col] = X[col].cat.codes

    for col in onehot_cols:
        has_na = X[col].isna().any()
        X = pd.get_dummies(X, columns=[col], dummy_na=has_na)

    # 5) 안전망: 예상 밖 NaN이 남아있으면 조용히 숨기지 않고 출력한 뒤 0으로 채운다
    remaining_na = X.isna().sum()
    remaining_na = remaining_na[remaining_na > 0]
    if len(remaining_na) > 0:
        print(f"  [경고] 명시적으로 처리되지 않은 NaN 컬럼 발견 — 0으로 채우고 계속 진행합니다:")
        print(remaining_na)
        X = X.fillna(0)

    return X


_fill_values_full = train_df[NAN_FILL_COLS].median()
print("NaN 대체 median 값:")
print(_fill_values_full)


NaN 대체 median 값:
사용자평균사용액_확장        27775.739501
사용자표준편차_확장        128056.821605
거래금액_Zscore_확장        -0.146994
dtype: float64


## 4. Tier 0 / Tier 0+1 두 피처셋으로 각각 5-fold 평가

베이스라인과 동일한 하이퍼파라미터(`n_estimators=200`, `max_samples='auto'`, `contamination='auto'`)로,
같은 fold 루프를 두 피처셋에 대해 각각 실행한다. 결과를 나중에 하나의 함수로 재사용할 수 있도록
`run_fold_cv()`로 묶어둔다.

In [4]:
N_SPLITS = 5
TOP_K_FRACTIONS = [0.01, 0.05, 0.10]


def run_fold_cv(tier_cols, label, n_estimators=200, max_samples='auto'):
    X_full = build_model_matrix(train_df, tier_cols, _fill_values_full)
    print(f"\n[{label}] 모델 입력 행렬 shape: {X_full.shape}")
    assert X_full.isna().sum().sum() == 0, f"[{label}] 결측이 남아있습니다."

    fold_rows = []
    for f in range(N_SPLITS):
        fold_train_idx = train_df.index[train_df['fold'] != f]
        fold_valid_idx = train_df.index[train_df['fold'] == f]

        X_tr = X_full.loc[fold_train_idx]
        X_va = X_full.loc[fold_valid_idx]
        y_va = train_df.loc[fold_valid_idx, '이상거래여부'].values

        iso = IsolationForest(
            n_estimators=n_estimators,
            max_samples=max_samples,
            contamination='auto',
            random_state=42,
            n_jobs=-1,
        )
        iso.fit(X_tr)
        anomaly_score = -iso.decision_function(X_va)

        pr_auc = average_precision_score(y_va, anomaly_score)
        row = {'fold': f, 'pr_auc': pr_auc}

        order = np.argsort(-anomaly_score)
        n_pos_total = y_va.sum()
        for frac in TOP_K_FRACTIONS:
            k = max(1, int(len(y_va) * frac))
            top_k_idx = order[:k]
            row[f'recall@top{int(frac*100)}%'] = y_va[top_k_idx].sum() / n_pos_total if n_pos_total > 0 else np.nan
            row[f'precision@top{int(frac*100)}%'] = y_va[top_k_idx].sum() / k

        fold_rows.append(row)
        print(f"  [{label} | fold {f}] PR-AUC={pr_auc:.4f}  recall@top5%={row['recall@top5%']:.3f}")

    return pd.DataFrame(fold_rows)


tier0_cols = FEATURE_TIERS['tier0_transaction_safe']
tier01_cols = FEATURE_TIERS['tier0_transaction_safe'] + FEATURE_TIERS['tier1_card_settlement_meta']

results_tier0 = run_fold_cv(tier0_cols, 'Tier0')
results_tier01 = run_fold_cv(tier01_cols, 'Tier0+1')



[Tier0] 모델 입력 행렬 shape: (1482969, 21)
  [Tier0 | fold 0] PR-AUC=0.4483  recall@top5%=0.606
  [Tier0 | fold 1] PR-AUC=0.4824  recall@top5%=0.620
  [Tier0 | fold 2] PR-AUC=0.5137  recall@top5%=0.644
  [Tier0 | fold 3] PR-AUC=0.5167  recall@top5%=0.649
  [Tier0 | fold 4] PR-AUC=0.3946  recall@top5%=0.632
  [경고] 명시적으로 처리되지 않은 NaN 컬럼 발견 — 0으로 채우고 계속 진행합니다:
카드이용한도금액    152
dtype: int64

[Tier0+1] 모델 입력 행렬 shape: (1482969, 32)
  [Tier0+1 | fold 0] PR-AUC=0.2005  recall@top5%=0.379
  [Tier0+1 | fold 1] PR-AUC=0.2200  recall@top5%=0.387
  [Tier0+1 | fold 2] PR-AUC=0.2066  recall@top5%=0.351
  [Tier0+1 | fold 3] PR-AUC=0.2274  recall@top5%=0.392
  [Tier0+1 | fold 4] PR-AUC=0.1962  recall@top5%=0.347


## 5. 결과 비교

In [5]:
summary_cols = ['pr_auc'] + [f'{m}@top{int(f*100)}%' for f in TOP_K_FRACTIONS for m in ['recall', 'precision']]

tier0_summary = results_tier0[summary_cols].mean().rename('Tier0 (14개)')
tier01_summary = results_tier01[summary_cols].mean().rename('Tier0+1 (25개)')

comparison = pd.concat([tier0_summary, tier01_summary], axis=1)
comparison['개선폭'] = comparison.iloc[:, 1] - comparison.iloc[:, 0]
comparison['개선율(%)'] = (comparison['개선폭'] / comparison.iloc[:, 0] * 100).round(2)

print("Tier0 fold별 표준편차 (노이즈 기준선):")
print(results_tier0[summary_cols].std().round(4))
print()
comparison.round(4)


Tier0 fold별 표준편차 (노이즈 기준선):
pr_auc              0.0510
recall@top1%        0.0227
precision@top1%     0.0854
recall@top5%        0.0178
precision@top5%     0.0138
recall@top10%       0.0110
precision@top10%    0.0043
dtype: float64



,Tier0 (14개),Tier0+1 (25개),개선폭,개선율(%)
pr_auc,0.4711,0.2101,-0.2610,-55.41
recall@top1%,0.1828,0.0859,-0.0969,-53.03
precision@top1%,0.6856,0.3220,-0.3635,-53.03
recall@top5%,0.6300,0.3713,-0.2587,-41.06
precision@top5%,0.4726,0.2785,-0.1940,-41.06
recall@top10%,0.7317,0.5465,-0.1852,-25.31
precision@top10%,0.2744,0.2049,-0.0694,-25.31


## 6. 중복 컬럼 제거 + Tier 1 개별 기여도 진단 (leave-one-in ablation)

**중복 제거**: `카드이용한도금액`(원본 금액)과 `카드이용한도금액_구간`(순서 코드)은 실제로 같은 정보를
두 번 담고 있다(전처리 4절 확인: 원본 값이 정확히 {0, 1백만, 5백만, 1천만} 4개뿐). 중복된 정보는 그 정보가
분기에 뽑힐 확률만 두 배로 늘릴 뿐 새로운 판별력을 주지 않으므로, **원본 금액은 빼고 구간 컬럼만 남긴다.**

**개별 기여도 진단**: Tier0+1 전체를 합쳤을 때 왜 성능이 크게 떨어지는지 확인하기 위해, Tier 1의 각 컬럼을
**하나씩만** Tier0에 추가해 성능 변화를 본다. 특정 컬럼 하나가 크게 깎아먹으면 그 컬럼이 범인이고, 전부
고르게 조금씩 깎인다면 Isolation Forest의 무작위 피처 선택 특성상 생기는 일반적인 희석 효과로 봐야 한다.

속도를 위해 이 진단은 fold 0~2(3개)만 사용한다.

In [6]:
TIER1_DEDUP = [c for c in FEATURE_TIERS['tier1_card_settlement_meta'] if c != '카드이용한도금액']
print(f"중복 제거 후 Tier 1: {len(TIER1_DEDUP)}개")
print(TIER1_DEDUP)

ABLATION_FOLDS = [0, 1, 2]
ablation_rows = []

# 기준선: Tier0만 (이미 4절에서 5-fold로 구했지만, 공정 비교를 위해 여기서도 동일 3-fold로 다시 잰다)
baseline_3fold = []
X_tier0 = build_model_matrix(train_df, tier0_cols, _fill_values_full)
for f in ABLATION_FOLDS:
    fold_train_idx = train_df.index[train_df['fold'] != f]
    fold_valid_idx = train_df.index[train_df['fold'] == f]
    iso = IsolationForest(n_estimators=200, max_samples='auto', contamination='auto', random_state=42, n_jobs=-1)
    iso.fit(X_tier0.loc[fold_train_idx])
    score = -iso.decision_function(X_tier0.loc[fold_valid_idx])
    y_va = train_df.loc[fold_valid_idx, '이상거래여부'].values
    baseline_3fold.append(average_precision_score(y_va, score))

baseline_pr_auc_3fold = float(np.mean(baseline_3fold))
print(f"\nTier0 단독 (3-fold 기준) PR-AUC: {baseline_pr_auc_3fold:.4f}\n")

for col in TIER1_DEDUP:
    trial_cols = tier0_cols + [col]
    X_trial = build_model_matrix(train_df, trial_cols, _fill_values_full)

    pr_aucs = []
    for f in ABLATION_FOLDS:
        fold_train_idx = train_df.index[train_df['fold'] != f]
        fold_valid_idx = train_df.index[train_df['fold'] == f]
        iso = IsolationForest(n_estimators=200, max_samples='auto', contamination='auto', random_state=42, n_jobs=-1)
        iso.fit(X_trial.loc[fold_train_idx])
        score = -iso.decision_function(X_trial.loc[fold_valid_idx])
        y_va = train_df.loc[fold_valid_idx, '이상거래여부'].values
        pr_aucs.append(average_precision_score(y_va, score))

    pr_auc_mean = float(np.mean(pr_aucs))
    delta = pr_auc_mean - baseline_pr_auc_3fold
    ablation_rows.append({
        'tier1_컬럼': col,
        'pr_auc_mean_3fold': pr_auc_mean,
        'Tier0 대비 변화': delta,
        '변화율(%)': round(delta / baseline_pr_auc_3fold * 100, 2),
    })
    print(f"  Tier0 + {col:<15} → PR-AUC={pr_auc_mean:.4f}  (변화 {delta:+.4f}, {delta/baseline_pr_auc_3fold*100:+.1f}%)")

ablation_df = pd.DataFrame(ablation_rows).sort_values('pr_auc_mean_3fold', ascending=False).reset_index(drop=True)
ablation_df


중복 제거 후 Tier 1: 10개
['개인법인구분코드_회원', '국내해외여부', '카드이용한도금액_구간', '법인카드_한도미기재추정', '승인거래코드', '승인발생경로코드', '로고구분코드', '일시불할부구분코드', '카드구분코드', '할부가능개월수']

Tier0 단독 (3-fold 기준) PR-AUC: 0.4815

  Tier0 + 개인법인구분코드_회원     → PR-AUC=0.4248  (변화 -0.0567, -11.8%)
  Tier0 + 국내해외여부          → PR-AUC=0.4594  (변화 -0.0221, -4.6%)
  Tier0 + 카드이용한도금액_구간     → PR-AUC=0.4538  (변화 -0.0277, -5.7%)
  Tier0 + 법인카드_한도미기재추정    → PR-AUC=0.4250  (변화 -0.0565, -11.7%)
  Tier0 + 승인거래코드          → PR-AUC=0.4643  (변화 -0.0172, -3.6%)
  Tier0 + 승인발생경로코드        → PR-AUC=0.4282  (변화 -0.0533, -11.1%)
  Tier0 + 로고구분코드          → PR-AUC=0.4637  (변화 -0.0178, -3.7%)
  Tier0 + 일시불할부구분코드       → PR-AUC=0.5160  (변화 +0.0345, +7.2%)
  Tier0 + 카드구분코드          → PR-AUC=0.4255  (변화 -0.0560, -11.6%)
  Tier0 + 할부가능개월수         → PR-AUC=0.4412  (변화 -0.0403, -8.4%)


,tier1_컬럼,pr_auc_mean_3fold,Tier0 대비 변화,변화율(%)
0,일시불할부구분코드,0.515978,0.034485,7.16
1,승인거래코드,0.464286,-0.017207,-3.57
2,로고구분코드,0.463698,-0.017795,-3.70
3,국내해외여부,0.459440,-0.022054,-4.58
4,카드이용한도금액_구간,0.453836,-0.027657,-5.74
5,할부가능개월수,0.441203,-0.040290,-8.37
6,승인발생경로코드,0.428218,-0.053276,-11.06
7,카드구분코드,0.425511,-0.055982,-11.63
8,법인카드_한도미기재추정,0.424993,-0.056501,-11.73
9,개인법인구분코드_회원,0.424811,-0.056683,-11.77


## 7. 진단 결과를 바탕으로 최종 Tier0+1 피처셋 확정

6절 결과에서 **Tier0 단독보다 확실히 낮게 나온 컬럼**은 제외 후보로 표시하고, 나머지(중립~긍정 기여)만
모아 최종 Tier0+1 조합을 다시 구성한다. "확실히 낮다"의 기준은 3-fold 노이즈 폭(하이퍼파라미터 탐색 때
관찰된 표준편차 0.02~0.04 수준)보다 뚜렷하게 큰 하락으로 잡는다.

In [7]:
DROP_THRESHOLD = -0.02  # Tier0 대비 이보다 더 크게 떨어지면 제외 후보 (3-fold 노이즈 폭 감안)

drop_candidates = ablation_df[ablation_df['Tier0 대비 변화'] < DROP_THRESHOLD]['tier1_컬럼'].tolist()
keep_candidates = [c for c in TIER1_DEDUP if c not in drop_candidates]

print(f"제외 후보 ({len(drop_candidates)}개): {drop_candidates}")
print(f"유지 후보 ({len(keep_candidates)}개): {keep_candidates}")

final_tier01_cols = tier0_cols + keep_candidates
X_final = build_model_matrix(train_df, final_tier01_cols, _fill_values_full)
print(f"\n최종 Tier0+1(정제) 입력 행렬 shape: {X_final.shape}")

final_fold_rows = []
for f in range(N_SPLITS):
    fold_train_idx = train_df.index[train_df['fold'] != f]
    fold_valid_idx = train_df.index[train_df['fold'] == f]
    X_tr = X_final.loc[fold_train_idx]
    X_va = X_final.loc[fold_valid_idx]
    y_va = train_df.loc[fold_valid_idx, '이상거래여부'].values

    iso = IsolationForest(n_estimators=200, max_samples='auto', contamination='auto', random_state=42, n_jobs=-1)
    iso.fit(X_tr)
    anomaly_score = -iso.decision_function(X_va)

    pr_auc = average_precision_score(y_va, anomaly_score)
    row = {'fold': f, 'pr_auc': pr_auc}
    order = np.argsort(-anomaly_score)
    n_pos_total = y_va.sum()
    for frac in TOP_K_FRACTIONS:
        k = max(1, int(len(y_va) * frac))
        top_k_idx = order[:k]
        row[f'recall@top{int(frac*100)}%'] = y_va[top_k_idx].sum() / n_pos_total if n_pos_total > 0 else np.nan
        row[f'precision@top{int(frac*100)}%'] = y_va[top_k_idx].sum() / k
    final_fold_rows.append(row)
    print(f"[fold {f}] PR-AUC={pr_auc:.4f}  recall@top5%={row['recall@top5%']:.3f}")

results_tier01_refined = pd.DataFrame(final_fold_rows)


제외 후보 (7개): ['국내해외여부', '카드이용한도금액_구간', '할부가능개월수', '승인발생경로코드', '카드구분코드', '법인카드_한도미기재추정', '개인법인구분코드_회원']
유지 후보 (3개): ['승인거래코드', '로고구분코드', '일시불할부구분코드']

최종 Tier0+1(정제) 입력 행렬 shape: (1482969, 24)
[fold 0] PR-AUC=0.4684  recall@top5%=0.577
[fold 1] PR-AUC=0.4932  recall@top5%=0.595
[fold 2] PR-AUC=0.5382  recall@top5%=0.633
[fold 3] PR-AUC=0.5338  recall@top5%=0.628
[fold 4] PR-AUC=0.3730  recall@top5%=0.568


## 8. 최종 비교 — Tier0 vs Tier0+1(정제 전) vs Tier0+1(정제 후)

In [8]:
tier0_summary_final = results_tier0[summary_cols].mean().rename('Tier0 (14개)')
tier01_raw_summary = results_tier01[summary_cols].mean().rename('Tier0+1 정제 전')
tier01_refined_summary = results_tier01_refined[summary_cols].mean().rename('Tier0+1 정제 후')

final_comparison = pd.concat([tier0_summary_final, tier01_raw_summary, tier01_refined_summary], axis=1)
final_comparison.round(4)


,Tier0 (14개),Tier0+1 정제 전,Tier0+1 정제 후
pr_auc,0.4711,0.2101,0.4813
recall@top1%,0.1828,0.0859,0.1936
precision@top1%,0.6856,0.3220,0.7264
recall@top5%,0.6300,0.3713,0.6002
precision@top5%,0.4726,0.2785,0.4502
recall@top10%,0.7317,0.5465,0.7321
precision@top10%,0.2744,0.2049,0.2746


## 9. `일시불할부구분코드` 단독 검증 (5-fold 전체)

6절 ablation(3-fold)에서 Tier 1 10개 중 유일하게 양의 신호(+7.2%)를 보인 컬럼이다. 3-fold 노이즈였는지
확인하기 위해, **Tier0 + 이 컬럼 1개**만으로 4절과 동일한 5-fold 전체 평가를 다시 돌린다. 나머지 9개
Tier1 컬럼은 개별로도 마이너스였으므로 이 검증에서 제외한다.

In [9]:
results_tier0_installment = run_fold_cv(
    tier0_cols + ['일시불할부구분코드'], 'Tier0+일시불할부구분코드'
)



[Tier0+일시불할부구분코드] 모델 입력 행렬 shape: (1482969, 22)
  [Tier0+일시불할부구분코드 | fold 0] PR-AUC=0.4705  recall@top5%=0.586
  [Tier0+일시불할부구분코드 | fold 1] PR-AUC=0.5082  recall@top5%=0.610
  [Tier0+일시불할부구분코드 | fold 2] PR-AUC=0.5692  recall@top5%=0.658
  [Tier0+일시불할부구분코드 | fold 3] PR-AUC=0.5336  recall@top5%=0.631
  [Tier0+일시불할부구분코드 | fold 4] PR-AUC=0.3914  recall@top5%=0.593


## 10. 최종 판단 — Tier0 vs Tier0+일시불할부구분코드

In [10]:
tier0_final = results_tier0[summary_cols].mean().rename('Tier0 (14개)')
tier0_installment_final = results_tier0_installment[summary_cols].mean().rename('Tier0+일시불할부구분코드 (15개)')

installment_comparison = pd.concat([tier0_final, tier0_installment_final], axis=1)
installment_comparison['개선폭'] = installment_comparison.iloc[:, 1] - installment_comparison.iloc[:, 0]
installment_comparison['개선율(%)'] = (
    installment_comparison['개선폭'] / installment_comparison.iloc[:, 0] * 100
).round(2)

print("Tier0 fold별 표준편차 (노이즈 기준선, 참고용):")
print(results_tier0[summary_cols].std().round(4))
print()
installment_comparison.round(4)


Tier0 fold별 표준편차 (노이즈 기준선, 참고용):
pr_auc              0.0510
recall@top1%        0.0227
precision@top1%     0.0854
recall@top5%        0.0178
precision@top5%     0.0138
recall@top10%       0.0110
precision@top10%    0.0043
dtype: float64



,Tier0 (14개),Tier0+일시불할부구분코드 (15개),개선폭,개선율(%)
pr_auc,0.4711,0.4946,0.0235,4.98
recall@top1%,0.1828,0.1948,0.0121,6.60
precision@top1%,0.6856,0.7308,0.0453,6.60
recall@top5%,0.6300,0.6154,-0.0147,-2.33
precision@top5%,0.4726,0.4616,-0.0110,-2.33
recall@top10%,0.7317,0.7508,0.0192,2.62
precision@top10%,0.2744,0.2816,0.0072,2.62


## 요약

**비교 방법**: 동일한 fold(카드 단위 5-fold), 동일한 하이퍼파라미터(n_estimators=200, max_samples='auto')로
Tier 0(14개)와 Tier 0+1(25개)을 각각 5회씩 학습·평가해 PR-AUC·top-k recall/precision을 비교했다.

**판단 기준**: 5절 출력의 `개선율(%)`이 Tier0의 fold 표준편차(노이즈 수준, 대략 PR-AUC 기준 ±10% 내외)보다
뚜렷하게 크면 "Tier 1 추가가 실질적으로 유의미하다"고 볼 수 있고, 그 이하면(하이퍼파라미터 튜닝 때처럼)
노이즈 범위 안이라 raw_payload에 Tier 1을 반드시 포함시켜야 한다는 강한 근거로 쓰기 어렵다.

**9~10절: `일시불할부구분코드` 단독 검증**
- 6절 3-fold ablation에서 유일하게 양의 신호(+7.2%)를 보인 컬럼을 5-fold 전체로 재검증했다. 노이즈였는지,
  실제로 유의미한 개선인지는 10절 `installment_comparison`과 Tier0 fold 표준편차를 비교해 판단한다.

**6~8절 진단 결과 반영**
- `카드이용한도금액`(원본)과 `카드이용한도금액_구간`(순서 코드)이 완전히 중복된 정보라 원본은 제외했다.
- Tier 1 10개 컬럼을 하나씩 Tier0에 추가해봄으로써, 어느 컬럼이 성능을 깎아먹는지(또는 기여하는지) 개별
  확인했다(6절 `ablation_df`). 뚜렷하게 하락시키는 컬럼은 제외하고 나머지만 모아 최종 Tier0+1(정제) 조합을
  다시 구성해 5-fold로 재평가했다(7~8절).

**다음 결정**
- 개선폭이 크다 → raw_payload 스키마 논의(Open Issue #6-7)에 "카드 메타데이터 포함 필요"라는 정량적 근거로 제시
- 개선폭이 작다 → Tier 0만으로 MVP를 시작하고, raw_payload 확정 후 Tier 1을 재평가하는 쪽으로 결정
- 어느 쪽이든 이번 결과를 팀 문서에 남겨, 나중에 "왜 이 피처셋을 선택했는지" 근거로 활용
